# Two-Stage Sexism Detection Pipeline
### UNICC Project — SIMBIG Conference Paper

**Stage 1 — Binary Classifier:** Detects whether a tweet is sexist or not (full dataset, 44,352 rows)  
**Stage 2 — Multi-class Classifier:** Categorizes sexist tweets into 4 categories (4,857 labeled rows)

**Inference flow:**
```
Tweet → Stage 1 → [not sexist: STOP] or [sexist] → Stage 2 → category
```

**Multi-class taxonomy (4 classes):**
- 0: Sexual Insults & Objectification
- 1: Gendered Stereotypes & Insults *(Victim Blaming merged in for balance)*
- 2: General Hostile Language
- 3: Threats & Physical Harm

---
**To swap models:** Change `STAGE1_MODEL` or `STAGE2_MODEL` in the config cell.

## 0. Setup & Installation

In [ ]:
!pip install transformers datasets scikit-learn torch accelerate -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuration
Change model names here to benchmark different models. Everything else adapts automatically.

In [ ]:
# ─────────────────────────────────────────────
# MODEL SELECTION — change these to swap models
# ─────────────────────────────────────────────
# Stage 1 & 2 options:
# 'dccuchile/bert-base-spanish-wwm-cased'       # BETO
# 'PlanTL-GOB-ES/roberta-base-bne'              # BERTIN
# 'cardiffnlp/twitter-xlm-roberta-base'         # XLM-T
# 'microsoft/mdeberta-v3-base'                  # mDeBERTa-v3
# 'xlm-roberta-large'                           # XLM-RoBERTa-large
# 'pysentimiento/robertuito-base-uncased'       # RoBERTuito

STAGE1_MODEL = 'pysentimiento/robertuito-base-uncased'   # XLM-T (best binary so far)
STAGE2_MODEL = 'pysentimiento/robertuito-base-uncased'  # RoBERTuito (best multiclass so far)

# ─────────────────────────────────────────────
# MOUNT GOOGLE DRIVE & SET DATA PATH
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/Copy of traducido_dataset_unificado_v_30jun - traducido_dataset_unificado_v_30jun (1).csv'  # update this

# ─────────────────────────────────────────────
# HYPERPARAMETERS
# ─────────────────────────────────────────────
MAX_LEN       = 128
BATCH_SIZE    = 16
EPOCHS        = 4
LEARNING_RATE = 2e-5
SEED          = 42

print('Stage 1 model:', STAGE1_MODEL)
print('Stage 2 model:', STAGE2_MODEL)

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Total rows: {len(df)}')

In [ ]:
# ── STAGE 1: Binary dataset ──────────────────────────────────────────────────
# target_column is the unified binary label across all source datasets (0=not sexist, 1=sexist)

binary_df = df[df['text_spanish'].notna() & df['target_column'].notna()].copy()
binary_df['label'] = binary_df['target_column'].astype(int)
binary_df = binary_df[['text_spanish', 'label']].rename(columns={'text_spanish': 'text'}).dropna()

print(f'Stage 1 dataset size: {len(binary_df)}')
print('Class distribution (0=not sexist, 1=sexist):')
print(binary_df['label'].value_counts())

In [ ]:
# ── STAGE 2: Multi-class dataset ─────────────────────────────────────────────
# Merge Class 5 (Victim Blaming) into Class 2 (Gendered Stereotypes)
# Rationale: both involve justifying/minimizing harm to women

LABEL_MERGE = {
    '1. Sexual Insults & Objectification': 'Sexual Insults & Objectification',
    '2. Gendered Stereotypes & Insults':   'Gendered Stereotypes & Insults',
    '3. General Hostile Language':         'General Hostile Language',
    '4. Threats & Physical Harm':          'Threats & Physical Harm',
    '5. Victim Blaming & Justification':   'Gendered Stereotypes & Insults'  # merged
}

MULTICLASS_LABELS = [
    'Sexual Insults & Objectification',
    'Gendered Stereotypes & Insults',
    'General Hostile Language',
    'Threats & Physical Harm'
]
label2id = {l: i for i, l in enumerate(MULTICLASS_LABELS)}
id2label = {i: l for l, i in label2id.items()}

mc_df = df[
    df['multiclass_final'].notna() &
    df['text_spanish'].notna() &
    (df['target_column'] == 1)
].copy()

mc_df['category'] = mc_df['multiclass_final'].map(LABEL_MERGE)
mc_df['label'] = mc_df['category'].map(label2id)
mc_df = mc_df[['text_spanish', 'label', 'category']].rename(columns={'text_spanish': 'text'}).dropna()

print(f'Stage 2 dataset size: {len(mc_df)}')
print('Class distribution after merge:')
print(mc_df['category'].value_counts())

## 3. Train / Validation / Test Splits

In [ ]:
def make_splits(df, seed=SEED):
    """70% train, 15% val, 15% test — stratified by label."""
    train, temp = train_test_split(df, test_size=0.3, random_state=seed, stratify=df['label'])
    val, test   = train_test_split(temp, test_size=0.5, random_state=seed, stratify=temp['label'])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

s1_train, s1_val, s1_test = make_splits(binary_df)
s2_train, s2_val, s2_test = make_splits(mc_df)

print(f'Stage 1 — Train: {len(s1_train)} | Val: {len(s1_val)} | Test: {len(s1_test)}')
print(f'Stage 2 — Train: {len(s2_train)} | Val: {len(s2_val)} | Test: {len(s2_test)}')

## 4. Shared Helper Functions

In [ ]:
def tokenize_data(tokenizer, df, max_len=MAX_LEN):
    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=max_len)
    dataset = Dataset.from_pandas(df[['text', 'label']])
    return dataset.map(tokenize, batched=True)


def get_class_weights(labels):
    classes = np.unique(labels)
    weights = compute_class_weight('balanced', classes=classes, y=labels)
    return torch.tensor(weights, dtype=torch.float)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    return {
        'macro_f1':    report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score'],
        'accuracy':    report['accuracy']
    }


def make_weighted_trainer(class_weights):
    """Returns a Trainer subclass using weighted cross-entropy loss."""
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop('labels')
            outputs = model(**inputs)
            loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
            loss = loss_fn(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss
    return WeightedTrainer


def get_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to='none'
    )


print('Helper functions ready.')

---
## STAGE 1 — Binary Classifier
**Data:** Full dataset (44,352 rows) | **Model:** `STAGE1_MODEL`

In [ ]:
print(f'Loading tokenizer: {STAGE1_MODEL}')
s1_tokenizer = AutoTokenizer.from_pretrained(STAGE1_MODEL)
s1_train_ds = tokenize_data(s1_tokenizer, s1_train)
s1_val_ds   = tokenize_data(s1_tokenizer, s1_val)
s1_test_ds  = tokenize_data(s1_tokenizer, s1_test)
print('Tokenization complete.')

In [ ]:
s1_weights = get_class_weights(s1_train['label'].values)
print(f'Class weights: {s1_weights}  (0=not sexist, 1=sexist)')

s1_model = AutoModelForSequenceClassification.from_pretrained(
    STAGE1_MODEL,
    num_labels=2,
    id2label={0: 'not sexist', 1: 'sexist'},
    label2id={'not sexist': 0, 'sexist': 1},
    ignore_mismatched_sizes=True
)

S1Trainer = make_weighted_trainer(s1_weights)
s1_trainer = S1Trainer(
    model=s1_model,
    args=get_training_args('./stage1_checkpoints'),
    train_dataset=s1_train_ds,
    eval_dataset=s1_val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Training Stage 1...')
s1_trainer.train()

In [ ]:
s1_preds = s1_trainer.predict(s1_test_ds)
s1_pred_labels = np.argmax(s1_preds.predictions, axis=-1)
s1_true_labels = s1_test_ds['label']

print('=' * 55)
print('STAGE 1 RESULTS —', STAGE1_MODEL)
print('=' * 55)
print(classification_report(s1_true_labels, s1_pred_labels,
      target_names=['not sexist', 'sexist'], zero_division=0))

In [ ]:
s1_trainer.save_model('./stage1_model')
s1_tokenizer.save_pretrained('./stage1_model')
print('Stage 1 model saved to ./stage1_model')

---
## STAGE 2 — Multi-class Classifier
**Data:** 4,857 sexist rows | **Model:** `STAGE2_MODEL`

In [ ]:
print(f'Loading tokenizer: {STAGE2_MODEL}')
s2_tokenizer = AutoTokenizer.from_pretrained(STAGE2_MODEL)
s2_train_ds = tokenize_data(s2_tokenizer, s2_train)
s2_val_ds   = tokenize_data(s2_tokenizer, s2_val)
s2_test_ds  = tokenize_data(s2_tokenizer, s2_test)
print('Tokenization complete.')

In [ ]:
s2_weights = get_class_weights(s2_train['label'].values)
print('Class weights:')
for i, w in enumerate(s2_weights):
    print(f'  {id2label[i]}: {w:.4f}')

s2_model = AutoModelForSequenceClassification.from_pretrained(
    STAGE2_MODEL,
    num_labels=len(MULTICLASS_LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

S2Trainer = make_weighted_trainer(s2_weights)
s2_trainer = S2Trainer(
    model=s2_model,
    args=get_training_args('./stage2_checkpoints'),
    train_dataset=s2_train_ds,
    eval_dataset=s2_val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Training Stage 2...')
s2_trainer.train()

In [ ]:
s2_preds = s2_trainer.predict(s2_test_ds)
s2_pred_labels = np.argmax(s2_preds.predictions, axis=-1)
s2_true_labels = s2_test_ds['label']

print('=' * 55)
print('STAGE 2 RESULTS —', STAGE2_MODEL)
print('=' * 55)
print(classification_report(s2_true_labels, s2_pred_labels,
      target_names=MULTICLASS_LABELS, zero_division=0))

In [ ]:
s2_trainer.save_model('./stage2_model')
s2_tokenizer.save_pretrained('./stage2_model')
print('Stage 2 model saved to ./stage2_model')

---
## End-to-End Pipeline Evaluation
Run both stages in sequence on the Stage 2 test set to simulate real-world deployment.

In [ ]:
def run_inference(texts, model, tokenizer, batch_size=32):
    model.eval()
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True,
                           max_length=MAX_LEN, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
    return np.array(all_preds)


s1_model.to(device)
s2_model.to(device)

test_texts = s2_test['text'].tolist()
test_true  = s2_test['label'].values

# Stage 1 filter
s1_binary = run_inference(test_texts, s1_model, s1_tokenizer)
print(f'Stage 1: {s1_binary.sum()} / {len(s1_binary)} tweets flagged as sexist')
print(f'Stage 1 false negatives (missed): {(s1_binary == 0).sum()}')

# Stage 2 on flagged tweets only
sexist_mask  = s1_binary == 1
sexist_texts = [t for t, m in zip(test_texts, sexist_mask) if m]
sexist_true  = test_true[sexist_mask]

if len(sexist_texts) > 0:
    s2_cats = run_inference(sexist_texts, s2_model, s2_tokenizer)
    print()
    print('=' * 55)
    print('END-TO-END PIPELINE RESULTS')
    print('=' * 55)
    print(classification_report(sexist_true, s2_cats,
          target_names=MULTICLASS_LABELS, zero_division=0))
else:
    print('No tweets passed Stage 1 — check model.')

## Results Summary Table

In [ ]:
def summarize(y_true, y_pred, stage, model_name):
    return {
        'Stage':       stage,
        'Model':       model_name,
        'Accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'Macro F1':    round(f1_score(y_true, y_pred, average='macro',    zero_division=0), 4),
        'Weighted F1': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'Precision':   round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'Recall':      round(recall_score(y_true, y_pred, average='macro',    zero_division=0), 4),
    }

rows = [
    summarize(s1_true_labels, s1_pred_labels, 'Stage 1 (Binary)',       STAGE1_MODEL),
    summarize(s2_true_labels, s2_pred_labels, 'Stage 2 (Multi-class)',  STAGE2_MODEL),
]
pd.DataFrame(rows)

## Inference on New Tweets

In [ ]:
def classify_tweet(text):
    """Run the full two-stage pipeline on a single Spanish tweet."""
    for model in [s1_model, s2_model]:
        model.eval()

    inputs = s1_tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        pred = torch.argmax(s1_model(**inputs).logits, dim=-1).item()
    if pred == 0:
        return {'binary': 'not sexist', 'category': None}

    inputs = s2_tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        cat = torch.argmax(s2_model(**inputs).logits, dim=-1).item()
    return {'binary': 'sexist', 'category': id2label[cat]}


examples = [
    'Las mujeres no deberian trabajar, su lugar es en la casa.',
    'Hoy fue un dia muy productivo en el trabajo.',
    'Esa mujer es una prostituta y merece lo que le pasa.',
    'Me encanta la nueva pelicula de accion.',
]

for tweet in examples:
    result = classify_tweet(tweet)
    print(f'Tweet:    {tweet}')
    print(f'Binary:   {result["binary"]}')
    if result['category']:
        print(f'Category: {result["category"]}')
    print()